# COS30049 A2 - Round 2 data for the Qwen 3B writer (Kaggle CPU session)

Builds the data of round 2's Qwen fine-tune (`docs/KAGGLE_LM_PLAN.md`, section 4) from the project repository
([Defaul119/COS30049A2](https://github.com/Defaul119/COS30049A2)) at a fixed commit, on a CPU session, which uses
no GPU quota:

* **DS7 v2**: about 40,000 explanation pairs from DS1 and every DS2 address, plus the glossary
  (`python -m explorer.cli build-ds7 --out data/processed/ds7_v2 --settings ds7v2`). The build stops if a prompt
  of round 1's frozen test set would be trained on. Its held-out glossary questions go to `glossary_test.jsonl`.
* **DS6 subset**: 10,000 smol-smoltalk conversations, holding out the same 200 retention conversations as round 1
  (checked byte for byte).

Every output file's SHA-256 goes into `data_manifest.json`. The GPU notebook `cos30049a2-qwen3b-explain-v2` reads
this notebook's output, checks each file against it, and the manifest is committed to the repository.

**Settings (right-hand panel):** Accelerator **None**, **Internet on**. Then **Save Version -> Save & Run All
(Commit)**. It takes about half an hour.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "PLACEHOLDER"   # the code this build uses; recorded in data_manifest.json
# environment.yml's versions, so the saved scikit-learn models load as they were fitted
PINS = ["numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1", "scipy==1.17.1", "pyarrow==20.0.0", "joblib==1.4.2",
        "huggingface_hub==0.30.2", "tokenizers==0.21.1"]
RETENTION_SHA256 = "0f862f4692d1af224c836824a34748176f588dd5aec3eb3c569a6f8bd6794931"  # round 1's chat_retention.jsonl

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

In [ ]:
sh(sys.executable, "-m", "explorer.cli", "build-ds7", "--out", "data/processed/ds7_v2", "--settings", "ds7v2",
   log="build-ds7-v2.log")
sh(sys.executable, "-m", "lm.chat_subset", "--max-conversations", "10000", "--retention-start", "2500",
   "--out", "data/qwen_v2", log="chat-subset-v2.log")
got = sha256(SRC / "data" / "qwen_v2" / "chat_retention.jsonl")
assert got == RETENTION_SHA256, "the retention conversations differ from round 1's; stop and check"
print("retention conversations: same as round 1")

In [ ]:
# The data, the build's summary table and log, and a manifest the GPU notebook checks and the repository keeps.
files = sorted([*(SRC / "data" / "processed" / "ds7_v2").glob("*.json*"), *(SRC / "data" / "qwen_v2").glob("*.json*")])
extras = [SRC / "reports" / "tables" / "ds7v2_summary.csv", SRC / "reports" / "logs" / "build-ds7.log"]
for f in files + [e for e in extras if e.exists()]:
    dest = WORK / f.relative_to(SRC)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(f, dest)
manifest = {"commit": COMMIT, "built_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "python": sys.version.split()[0],
            "packages": {p.split("==")[0]: version(p.split("==")[0]) for p in PINS},
            "files": {f.relative_to(SRC).as_posix(): sha256(f) for f in files}}
(WORK / "data_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 2**20:9.1f} MB  {p.relative_to(WORK)}")